In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))
#         break

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session`

In [2]:
import torch
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [3]:
# frame = pd.read_csv("class_names.csv")
# names_classes = frame['name'].to_list()
# names_classes
import os
import pandas as pd
from pathlib import Path

train_dir = Path("./train")

# 1. Собираем уникальные названия классов из имён файлов
class_names = sorted(list(set(f.stem.split('_')[0] for f in train_dir.glob('*.jpg'))))

# 2. Создаём DataFrame с метками от 0 до 13
df_classes = pd.DataFrame({
    "name": class_names,
    "label": range(len(class_names))
})

# 3. Сохраняем в файл
df_classes.to_csv("class_names.csv", index=False)

print("Файл class_names.csv успешно создан!")
print(df_classes)

Файл class_names.csv успешно создан!
        name  label
0       bear      0
1       bird      1
2        cat      2
3       deer      3
4        dog      4
5    dolphin      5
6   elephant      6
7    giraffe      7
8      horse      8
9   kangaroo      9
10      lion     10
11     panda     11
12     tiger     12
13     zebra     13


In [8]:
MEAN = torch.tensor([0.485, 0.456, 0.406])
STD  = torch.tensor([0.229, 0.224, 0.225])

def imshow(images, nrow=8, figsize=(16, 12), mean=MEAN, std=STD):
    img = torchvision.utils.make_grid(images, nrow=nrow, padding=0).to(device)
    img = img.transpose(2, 0).transpose(0, 1)
    std = torch.as_tensor(std).to(device)
    mean = torch.as_tensor(mean).to(device)
    img = (img * std + mean)
    img = img.data.cpu().numpy()
    plt.figure(figsize=figsize)
    plt.imshow(img); plt.axis('off'); plt.show()

images, labels = next(iter(train_loader))     # <- train_loader, не trainloader
imshow(images)
print('first raw classes', ' '.join('%5s' % names_classes[labels[j]] for j in range(images.size(0))))
print(images.shape)


RuntimeError: DataLoader worker (pid(s) 4356, 16900) exited unexpectedly

In [7]:
import os, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision import transforms

# ----------------------------
# 0. Настройки
# ----------------------------
DATA_DIR = Path(".")          # в Kaggle: Path("/kaggle/input/имя-датасета")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test_new"
CLASS_CSV = DATA_DIR / "class_names.csv"

IMG_SIZE = 160                 # если есть GPU и позволяет время, можно 224
BATCH_SIZE = 64
EPOCHS = 12
LR = 3e-4
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

# ----------------------------
# 1. Классы
# ----------------------------
classes_df = pd.read_csv(CLASS_CSV)
print(classes_df.head())

class_to_idx = {
    str(row["name"]): int(row["label"])
    for _, row in classes_df.iterrows()
}
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)
print("num_classes =", num_classes)
print(class_to_idx)

# ----------------------------
# 2. Dataset
# ----------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class AnimalDataset(Dataset):
    def __init__(self, root, class_to_idx=None, transform=None):
        self.root = Path(root)
        self.class_to_idx = class_to_idx
        self.transform = transform

        self.paths = sorted([
            p for p in self.root.rglob("*")
            if p.is_file() and p.suffix.lower() in IMG_EXTS
        ])
        if len(self.paths) == 0:
            raise RuntimeError(f"Не найдено изображений в {self.root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.class_to_idx is not None:
            # имя файла: panda_001.jpg -> panda
            cls_name = path.stem.split("_")[0]
            label = self.class_to_idx[cls_name]
            return image, label
        else:
            return image, path.name

# ----------------------------
# 3. Аугментации
# ----------------------------
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ----------------------------
# 4. Train / validation split
# ----------------------------
full_train = AnimalDataset(TRAIN_DIR, class_to_idx=class_to_idx, transform=train_tf)

val_size = int(0.15 * len(full_train))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Для val нужен отдельный transform, поэтому подменим transform у val-части
# Проще: создать val_dataset на тех же путях с val_tf
class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path = self.subset.dataset.paths[self.subset.indices[idx]]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        cls_name = path.stem.split("_")[0]
        label = class_to_idx[cls_name]
        return image, label

val_ds = SubsetWithTransform(val_ds, val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

# ----------------------------
# 5. Модель
# ----------------------------
def make_model(num_classes):
    try:
        from torchvision.models import ResNet18_Weights
        model = torchvision.models.resnet18(weights=ResNet18_Weights.DEFAULT)
        print("Используются pretrained-веса ResNet18")
    except Exception as e:
        print("Pretrained-веса недоступны, обучение с нуля:", e)
        model = torchvision.models.resnet18(weights=None)

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = make_model(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ----------------------------
# 6. Обучение
# ----------------------------
best_acc = 0.0
best_path = "best_model.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    scheduler.step()

    train_loss /= train_total
    train_acc = train_correct / train_total

    # validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"  saved best model, val_acc={best_acc:.4f}")

print("Best val_acc =", best_acc)

# ----------------------------
# 7. Предсказание для test_new
# ----------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

test_ds = AnimalDataset(TEST_DIR, class_to_idx=None, transform=val_tf)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

rows = []
with torch.no_grad():
    for images, file_names in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()

        for file_name, pred in zip(file_names, preds):
            rows.append((file_name, int(pred)))

submission = pd.DataFrame(rows, columns=["FILE_NAME", "TARGET"])
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv сохранён, строк:", len(submission))


device = cpu
   name  label
0  bear      0
1  bird      1
2   cat      2
3  deer      3
4   dog      4
num_classes = 14
{'bear': 0, 'bird': 1, 'cat': 2, 'deer': 3, 'dog': 4, 'dolphin': 5, 'elephant': 6, 'giraffe': 7, 'horse': 8, 'kangaroo': 9, 'lion': 10, 'panda': 11, 'tiger': 12, 'zebra': 13}
Используются pretrained-веса ResNet18


D:\python_programming\for_git_repositories\work_folder\STUDY\2._CV2._Классификация\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: DataLoader worker (pid(s) 4468, 7308) exited unexpectedly